# 03 -- Gold Layer Analysis
**Project:** Airbnb Market Analysis -- New York City  
**Layer:** Gold (aggregated, business-ready)  
**Source:** workspace.default.silver_airbnb_listings  
**Author:** B. Sarang  

Applies window functions, feature engineering, and aggregations to identify top-performing neighbourhoods and room types. Writes Gold tables for downstream dashboarding.

## 1. Configuration

In [0]:
SILVER_TABLE = "workspace.default.silver_airbnb_listings"
GOLD_NEIGHBOURHOOD = "workspace.default.gold_neighbourhood_performance"
GOLD_ROOM_TYPE = "workspace.default.gold_room_type_summary"
GOLD_HOST = "workspace.default.gold_host_performance"

## 2. Read Silver table

In [0]:
df = spark.table(SILVER_TABLE)
print(f"Silver row count: {df.count()}")
display(df.limit(5))

## 3. Feature engineering

In [0]:
from pyspark.sql.functions import (
    col, round as spark_round, when, log1p,
    avg, count, stddev, min as spark_min, max as spark_max,
    dense_rank, percent_rank, ntile
)
from pyspark.sql.window import Window

df_features = (
    df
    # Estimated monthly revenue proxy
    .withColumn('est_monthly_revenue',
        spark_round(col('price') * col('reviews_per_month') * 3.5, 2))
    # Demand score: reviews per month normalised by availability
    .withColumn('demand_score',
        spark_round(
            when(col('availability_365') > 0,
                col('reviews_per_month') / (col('availability_365') / 365))
            .otherwise(None), 4))
    # Value score: rating per dollar
    .withColumn('value_score',
        spark_round(
            when(col('price') > 0,
                col('review_scores_rating') / col('price'))
            .otherwise(None), 6))
    # Log price for distribution analysis
    .withColumn('log_price', spark_round(log1p(col('price')), 4))
)

print("Feature engineering complete")
display(df_features.select('neighbourhood', 'room_type', 'price', 'est_monthly_revenue', 'demand_score', 'value_score').limit(10))

## 4. Window functions -- neighbourhood ranking

In [0]:
window_borough = Window.partitionBy('borough').orderBy(col('avg_rating').desc())
window_overall = Window.orderBy(col('avg_rating').desc())

df_neighbourhood = (
    df_features
    .groupBy('borough', 'neighbourhood')
    .agg(
        count('id').alias('total_listings'),
        spark_round(avg('price'), 2).alias('avg_price'),
        spark_round(avg('review_scores_rating'), 2).alias('avg_rating'),
        spark_round(avg('reviews_per_month'), 2).alias('avg_reviews_per_month'),
        spark_round(avg('availability_365'), 0).alias('avg_availability'),
        spark_round(avg('est_monthly_revenue'), 2).alias('avg_est_monthly_revenue'),
        spark_round(avg('demand_score'), 4).alias('avg_demand_score')
    )
)

df_neighbourhood = (
    df_neighbourhood
    .withColumn('rank_within_borough', dense_rank().over(window_borough))
    .withColumn('overall_rank', dense_rank().over(window_overall))
    .withColumn('rating_percentile', spark_round(percent_rank().over(window_overall) * 100, 1))
)

display(df_neighbourhood.orderBy('overall_rank').limit(20))

## 5. Room type summary

In [0]:
df_room_type = (
    df_features
    .groupBy('room_type')
    .agg(
        count('id').alias('total_listings'),
        spark_round(avg('price'), 2).alias('avg_price'),
        spark_round(spark_min('price'), 2).alias('min_price'),
        spark_round(spark_max('price'), 2).alias('max_price'),
        spark_round(stddev('price'), 2).alias('stddev_price'),
        spark_round(avg('review_scores_rating'), 2).alias('avg_rating'),
        spark_round(avg('reviews_per_month'), 2).alias('avg_reviews_per_month'),
        spark_round(avg('availability_365'), 0).alias('avg_availability'),
        spark_round(avg('est_monthly_revenue'), 2).alias('avg_est_monthly_revenue')
    )
    .orderBy('avg_price', ascending=False)
)

display(df_room_type)

## 6. Host performance -- superhost vs non-superhost

In [0]:
window_host = Window.orderBy(col('total_revenue_proxy').desc())

df_host = (
    df_features
    .groupBy('host_id', 'host_name', 'host_is_superhost')
    .agg(
        count('id').alias('total_listings'),
        spark_round(avg('price'), 2).alias('avg_price'),
        spark_round(avg('review_scores_rating'), 2).alias('avg_rating'),
        spark_round(avg('reviews_per_month'), 2).alias('avg_reviews_per_month'),
        spark_round(avg('est_monthly_revenue') * count('id'), 2).alias('total_revenue_proxy')
    )
    .withColumn('host_revenue_rank', dense_rank().over(window_host))
    .filter(col('total_listings') >= 2)
    .orderBy('host_revenue_rank')
)

display(df_host.limit(20))

## 7. Subquery -- top neighbourhood per borough

In [0]:
spark.sql(f"""
    SELECT borough, neighbourhood, avg_rating, avg_price, total_listings
    FROM (
        SELECT
            borough,
            neighbourhood,
            ROUND(AVG(review_scores_rating), 2) AS avg_rating,
            ROUND(AVG(price), 2) AS avg_price,
            COUNT(id) AS total_listings,
            DENSE_RANK() OVER (PARTITION BY borough ORDER BY AVG(review_scores_rating) DESC) AS rank
        FROM {SILVER_TABLE}
        WHERE review_scores_rating IS NOT NULL
        GROUP BY borough, neighbourhood
        HAVING COUNT(id) >= 10
    ) ranked
    WHERE rank = 1
    ORDER BY avg_rating DESC
""").display()

## 8. Write Gold tables

In [0]:
for df_gold, table_name in [
    (df_neighbourhood, GOLD_NEIGHBOURHOOD),
    (df_room_type, GOLD_ROOM_TYPE),
    (df_host, GOLD_HOST)
]:
    (
        df_gold.write
        .format('delta')
        .mode('overwrite')
        .option('overwriteSchema', 'true')
        .saveAsTable(table_name)
    )
    print(f"Written: {table_name}")

## 9. Transaction log -- Gold neighbourhood table

In [0]:
spark.sql(f"DESCRIBE HISTORY {GOLD_NEIGHBOURHOOD}").display()